# Brain Tumor MRI Classification
### Master Notebook
### This notebook runs the complete pipeline end-to-end.
### Run each section in order.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
sys.path.insert(0, '/content/drive/MyDrive/brain_tumor_classification')

# Install dependencies
import subprocess
subprocess.run(
    ['pip', 'install', '-q', '-r',
     '/content/drive/MyDrive/brain_tumor_classification/requirements.txt'],
    capture_output=True
)

print("=" * 50)
print("  Environment ready.")
print("  IMPORTANT: Runtime → Restart session")
print("=" * 50)

Mounted at /content/drive
  Environment ready.
  IMPORTANT: Runtime → Restart session


In [1]:
# ============================================================
# Imports and configuration
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import sys, os
sys.path.insert(0, '/content/drive/MyDrive/brain_tumor_classification')

import torch
import numpy as np
import matplotlib.pyplot as plt

# Load project config
from config import get_default_config
cfg = get_default_config()

# Override data path for Colab
cfg.paths.data_dir  = '/content/drive/MyDrive/brain_tumor_classification/data'
cfg.paths.root      = '/content/drive/MyDrive/brain_tumor_classification'

# Verify GPU
print("=" * 50)
print(f"  PyTorch  : {torch.__version__}")
print(f"  CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU      : {torch.cuda.get_device_name(0)}")
print(f"  Backbone : {cfg.model.backbone}")
print(f"  Img size : {cfg.data.image_size}x{cfg.data.image_size}")
print(f"  Batch    : {cfg.data.batch_size}")
print("=" * 50)

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = cfg.paths.data_dir
print(f"Using device: {DEVICE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  PyTorch  : 2.10.0+cpu
  CUDA     : False
  Backbone : efficientnet_b3
  Img size : 224x224
  Batch    : 32
Using device: cpu


In [3]:
# ============================================================
# Data Pipeline
# ============================================================

# ── Load DataLoaders
from src.dataset import create_dataloaders, get_dataset_info

train_loader, val_loader, test_loader, info = create_dataloaders(
    data_dir    = DATA_DIR,
    image_size  = cfg.data.image_size,
    batch_size  = cfg.data.batch_size,
    val_split   = cfg.data.val_split,
    num_workers = cfg.data.num_workers,
    seed        = cfg.data.seed,
)

get_dataset_info(info)

# ── Sanity check: one batch
images, labels = next(iter(train_loader))
print(f"\n  Batch shape : {images.shape}")
print(f"  Label sample: {labels[:8].tolist()}")

  Dataset Summary
  Classes      : glioma, meningioma, notumor, pituitary
  Image size   : 224x224
  Batch size   : 32

  Train set    :  4480 images  (140 batches)
  Val set      :  1120 images  (35 batches)
  Test set     :  1600 images  (50 batches)
  Total        :  7200 images

  Class → Index mapping:
    0 → glioma
    1 → meningioma
    2 → notumor
    3 → pituitary

  Batch shape : torch.Size([32, 3, 224, 224])
  Label sample: [2, 2, 0, 3, 2, 2, 2, 1]
